# Chapter 23 — Markov Chains
*Tier 3: All Tracks*

## 🎯 Learning Objectives
- Define Markov chains and transition matrices
- Compute stationary distributions
- Apply to web ranking, weather, and recommendation systems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import networkx as nx

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — The Markov Property

A sequence $X_0, X_1, X_2, \ldots$ is a **Markov chain** if:

$$P(X_{n+1} = j \mid X_n = i, X_{n-1}, \ldots) = P(X_{n+1} = j \mid X_n = i) = P_{ij}$$

The **transition matrix** $P$ is a stochastic matrix:
$$P_{ij} \geq 0, \quad \sum_j P_{ij} = 1 \text{ for all } i$$

The **n-step distribution**: $\pi^{(n)} = \pi^{(0)} P^n$

In [ ]:
# Weather Markov chain: Sunny, Cloudy, Rainy
states = ["Sunny", "Cloudy", "Rainy"]
P = np.array([
    [0.70, 0.20, 0.10],   # from Sunny
    [0.30, 0.40, 0.30],   # from Cloudy
    [0.20, 0.35, 0.45],   # from Rainy
])

# Verify stochastic
assert np.allclose(P.sum(axis=1), 1), "Not a valid stochastic matrix"

# Visualise transition graph
G = nx.DiGraph()
G.add_nodes_from(states)
for i, s1 in enumerate(states):
    for j, s2 in enumerate(states):
        if P[i, j] > 0.05:
            G.add_edge(s1, s2, weight=P[i, j])

plt.figure(figsize=(7, 5))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(G, pos, node_size=2000, node_color="steelblue", alpha=0.8)
nx.draw_networkx_labels(G, pos, font_color="white", font_size=11)
edge_labels = {(u,v): f"{d['weight']:.2f}" for u,v,d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=9)
nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=20, connectionstyle="arc3,rad=0.1")
plt.title("Weather Markov Chain")
plt.axis("off"); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Stationary Distribution

The **stationary distribution** $\pi$ satisfies:
$$\pi P = \pi, \quad \sum_i \pi_i = 1$$

For ergodic chains: regardless of initial state, $\pi^{(n)} \to \pi$ as $n \to \infty$.

In [ ]:
# Find stationary distribution by eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(P.T)
idx = np.argmin(np.abs(eigenvalues - 1))
pi = np.real(eigenvectors[:, idx])
pi /= pi.sum()
print("Stationary distribution (eigenvector):")
for s, p in zip(states, pi):
    print(f"  P({s}) = {p:.4f}")

# Verify by power iteration
pi_current = np.array([1.0, 0.0, 0.0])  # start sunny
for n in range(100):
    pi_current = pi_current @ P
print("
Stationary distribution (power iteration after 100 steps):")
for s, p in zip(states, pi_current):
    print(f"  P({s}) = {p:.4f}")

## 3. Simulation — Convergence to Stationary Distribution

In [ ]:
n_steps = 200
n_chains = 5
initial_states = [0, 0, 1, 2, 2]
colors = ["blue", "red", "green", "purple", "orange"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, state_idx, sname in zip(axes, range(3), states):
    ax.axhline(pi[state_idx], color="black", lw=2, linestyle="--",
               label=f"Stationary π={pi[state_idx]:.3f}")
    for init, color in zip(initial_states, colors):
        dist = np.zeros((n_steps+1, 3))
        dist[0, init] = 1.0
        for n in range(1, n_steps+1):
            dist[n] = dist[n-1] @ P
        ax.plot(dist[:, state_idx], alpha=0.6, color=color, lw=1)
    ax.set_title(f"P({sname} at step n)"); ax.set_xlabel("Step")
    ax.legend(fontsize=9)
plt.suptitle("Convergence to Stationary Distribution", fontweight="bold")
plt.tight_layout(); plt.show()

## 4. Application — PageRank Algorithm

In [ ]:
# PageRank as Markov chain
# 5-page website
pages = ["Home", "About", "Blog", "Shop", "Contact"]
links = np.array([
    [0, 1, 1, 1, 1],
    [1, 0, 1, 0, 0],
    [1, 0, 0, 1, 0],
    [1, 0, 1, 0, 1],
    [1, 1, 0, 0, 0],
])

# Row-normalise to make transition matrix
row_sums = links.sum(axis=1, keepdims=True)
P_pages = links / row_sums

# PageRank with damping factor d=0.85
d = 0.85
n_pages = len(pages)
P_pagerank = d * P_pages + (1-d) * np.ones((n_pages, n_pages)) / n_pages

# Power iteration
rank = np.ones(n_pages) / n_pages
for _ in range(100):
    rank = rank @ P_pagerank
rank_sorted = sorted(zip(pages, rank), key=lambda x: -x[1])
print("PageRank scores:")
for page, r in rank_sorted:
    print(f"  {page:<10} {r:.4f}")

plt.barh([p for p, _ in rank_sorted], [r for _, r in rank_sorted], color="steelblue")
plt.xlabel("PageRank"); plt.title("PageRank Scores")
plt.tight_layout(); plt.show()

In [ ]:
# Practice: Absorbing Markov chain — Gambler's ruin
# States: 0 (broke), 1, 2, 3, 4, 5 (goal), with 0 and 5 absorbing
p_win = 0.45  # probability of winning each bet
states_gambler = list(range(6))
P_gambler = np.zeros((6, 6))
P_gambler[0, 0] = 1.0   # absorbing
P_gambler[5, 5] = 1.0   # absorbing
for i in range(1, 5):
    P_gambler[i, i+1] = p_win
    P_gambler[i, i-1] = 1 - p_win

# Simulate from starting position 2
n_sim = 10_000
wins = 0
for _ in range(n_sim):
    state = 2
    for _ in range(1000):
        state = rng.choice(6, p=P_gambler[state])
        if state == 0 or state == 5:
            break
    wins += (state == 5)
print(f"P(reach 5 | start at 2) ≈ {wins/n_sim:.4f}")
# Analytic: (1-(q/p)^i) / (1-(q/p)^N)
q = 1 - p_win
analytic = (1-(q/p_win)**2) / (1-(q/p_win)**5)
print(f"Analytic:                 {analytic:.4f}")